Quiz 47 — YouTube Channel Analytics  [SOLVED]
Difficulty: Hard

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(100)
views    = np.random.normal(8000, 2500, 52).clip(500, 50000).astype(float)
likes    = (views * np.random.uniform(0.04, 0.12, 52)).astype(int)
comments = (views * np.random.uniform(0.005, 0.02, 52)).astype(int)
views[30] = 95000

# ── 1. Z-score outlier detection ─────────────────────────────────────────────
z            = (views - views.mean()) / views.std()
viral_weeks  = np.where(np.abs(z) > 3)[0]
print("Viral week indices:", viral_weeks)

# ── 2. Clean stats ────────────────────────────────────────────────────────────
clean_views = np.delete(views, viral_weeks)
print(f"Clean  — Mean: {clean_views.mean():,.0f}  "
      f"Median: {np.median(clean_views):,.0f}  Std: {clean_views.std():,.0f}")

# ── 3. Engagement rate ────────────────────────────────────────────────────────
engagement  = (likes + comments) / views * 100
p90_eng     = np.percentile(engagement, 90)
high_eng_wk = np.where(engagement > p90_eng)[0]
print(f"\n90th pct engagement: {p90_eng:.2f}%")
print(f"High-engagement weeks ({len(high_eng_wk)}): {high_eng_wk + 1}")

# ── 4. Like ratio top-5 ───────────────────────────────────────────────────────
like_ratio  = likes / views
top5_lr_idx = np.argsort(like_ratio)[-5:][::-1]
print("\nTop-5 like ratio weeks:")
for i in top5_lr_idx:
    print(f"  Week {i+1:2d}: ratio={like_ratio[i]:.4f}  views={views[i]:,.0f}")

# ── 5. Plots ──────────────────────────────────────────────────────────────────
weeks = np.arange(1, 53)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Line chart
colours_line = ["red" if i in viral_weeks else "steelblue" for i in range(52)]
ax1.bar(weeks, views / 1000, color=colours_line, edgecolor="none")
ax1.set_title("Weekly Views (viral week in red)")
ax1.set_xlabel("Week")
ax1.set_ylabel("Views (thousands)")

# Scatter: views vs engagement, coloured by above/below median views
med_views = np.median(views)
sc_colours = np.where(views >= med_views, "green", "grey")
ax2.scatter(views / 1000, engagement, c=sc_colours, alpha=0.7)
ax2.set_title("Views vs Engagement Rate")
ax2.set_xlabel("Views (thousands)")
ax2.set_ylabel("Engagement Rate (%)")
ax2.axhline(p90_eng, color="red", linestyle="--", label=f"P90={p90_eng:.1f}%")
ax2.legend()

plt.tight_layout()
plt.savefig("hard_07_youtube_analytics.png", dpi=100)
plt.show()

# ── WHY ───────────────────────────────────────────────────────────────────────
# Z-score > 3 means the value is more than 3 standard deviations above the mean —
# an extremely rare occurrence in normal data.
# Engagement rate normalises by views, making small and large channels comparable.